# Attention, Clustering and Self-Supervision

Prepared for Fall 2026. Reference: the notes for Lecture 14 (Transformers for Vision) and, for the last section, Lecture 13; page numbers refer to the Lecture 14 notes unless Lecture 13 is named. If you find a bug, please email the TAs or tell us at office hours.

**Before you start.** Run the notebook from the homework folder, so that `data` is next to it. Every figure and number a question asks for must appear in your PDF: to vary a parameter, copy the cell below the question (or loop) rather than re-running it above.

---

Teal = normalised over the clusters, orange = over the pixels; a cyan square marks the query position $o$.

## Utilities

In [ ]:
from __future__ import annotations
import itertools
import os

import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patheffects as pe
from PIL import Image

%matplotlib inline

if not os.path.isdir("data/attention"):
    raise FileNotFoundError("The folder data/attention was not found. Start Jupyter in the homework folder, "
                            "so that the folder data is next to this notebook.")

TEAL = "#0F7B6C"      # normalised over the clusters (p. 45, 55, 58, 61)
ORANGE = "#E08214"    # normalised over the pixels (p. 43, 45, 56, 61)
CYAN = "#7FD6FF"      # the query position o (p. 6, Figure 2)


# ----------------------------------------------------------------------------------------------- images and tokens
def load_image(name: str) -> npt.NDArray:
    """The photograph data/attention/<name> as an array of shape (H, W, 3) with values in [0, 1]."""
    return np.asarray(Image.open(f"data/attention/{name}").convert("RGB"), dtype=float) / 255


def patch_tokens(image, size: int = 15, kind: str = "colour"):
    """Tokens x_p of an image cut into size x size patches (p. 6, Figure 2), and the shape (rows, cols) of the patch grid.

    kind = "colour": x_p is the mean colour of the patch, d = 3.
    kind = "grey":   x_p is the grey patch reduced to 5 x 5 by averaging blocks of pixels and flattened, d = 25.
    The tokens are centred (the mean token of the image is subtracted) and scaled to the common length sqrt(d), so
    that x_o^T x_p = d cos(angle between x_o and x_p) is largest for the most similar patch and its range grows with d.
    Pixels beyond the last full patch are dropped.
    """
    rows, cols = image.shape[0] // size, image.shape[1] // size
    blocks = image[:rows * size, :cols * size].reshape(rows, size, cols, size, 3).transpose(0, 2, 1, 3, 4)
    if kind == "colour":
        tokens = blocks.mean(axis=(2, 3)).reshape(rows * cols, 3)
    elif kind == "grey":
        b = size // 5
        if b == 0:
            raise ValueError("grey tokens need size >= 5")
        grey = blocks.mean(axis=4)[:, :, :5 * b, :5 * b]
        tokens = grey.reshape(rows, cols, 5, b, 5, b).mean(axis=(3, 5)).reshape(rows * cols, 25)
    else:
        raise ValueError(kind)
    tokens = tokens - tokens.mean(axis=0)
    d = tokens.shape[1]
    return np.sqrt(d) * tokens / (np.linalg.norm(tokens, axis=1, keepdims=True) + 1e-12), (rows, cols)


# ----------------------------------------------------------------------------------------------- self-attention
def attention_weights(Q, K, scale=True):
    """Attention weights a_op = exp(q_o^T k_p / sqrt(d)) / sum_p' exp(q_o^T k_p' / sqrt(d)) (p. 9): one row per
    query o, one column per key p, every row sums to 1.

    scale is the sqrt(d) scaling of p. 9: True divides the affinities q_o^T k_p by sqrt(d); False leaves them as
    they are, which gives the weights exp(q_o^T k_p) / sum_p' exp(q_o^T k_p') of p. 5-7; a positive number
    divides them by that number instead. The largest affinity of each row is subtracted before the exponential,
    which does not change the weights but prevents overflow.
    """
    affinities = Q @ K.T
    if scale is True:
        affinities = affinities / np.sqrt(Q.shape[1])
    elif scale is not False:
        if not scale > 0:
            raise ValueError("scale must be True, False or a positive number")
        affinities = affinities / scale
    affinities = affinities - affinities.max(axis=1, keepdims=True)
    weights = np.exp(affinities)
    return weights / weights.sum(axis=1, keepdims=True)


def attention_map(tokens, o, grid, scale=True):
    """The weights a_op of the query at grid position o = (row, col) over all positions p, as a rows x cols map.
    W^Q = W^K = identity, so q_o = x_o and k_p = x_p (p. 9); scale as in attention_weights."""
    rows, cols = grid
    r, c = int(o[0]), int(o[1])
    if (r, c) != (o[0], o[1]) or not (0 <= r < rows and 0 <= c < cols):
        raise ValueError(f"o = {o} is not a (row, col) position on the {rows} x {cols} grid: rows 0..{rows - 1}, "
                         f"columns 0..{cols - 1}; read positions off the axes of the figures")
    index = r * cols + c
    return attention_weights(tokens[index:index + 1], tokens, scale)[0].reshape(rows, cols)


def keys_for_90_percent(weights) -> int:
    """How many of the largest weights of one query are needed to reach 90 % of its total weight (which is 1)."""
    return int(np.searchsorted(np.cumsum(np.sort(np.ravel(weights))[::-1]), 0.9) + 1)


# ----------------------------------------------------------------------------------------------- k-means (Lecture 8)
def nearest_centre(X, centres):
    """The index of the nearest centre for every row of X, from |x_p - c_g|^2 = |x_p|^2 - 2 x_p^T c_g + |c_g|^2
    (ties go to the smaller index)."""
    d2 = (X ** 2).sum(1)[:, None] - 2 * X @ centres.T + (centres ** 2).sum(1)[None, :]
    return d2.argmin(axis=1)


def kmeans_update(X, assignment, centres):
    """The update step of k-means: every centre becomes the mean of its pixels, c_g = sum_p V_pg x_p / sum_p V_pg.
    An empty cluster is re-started at the pixel farthest from its own centre."""
    new = centres.copy()
    for g in range(len(centres)):
        members = assignment == g
        new[g] = X[members].mean(axis=0) if members.any() else X[((X - centres[assignment]) ** 2).sum(1).argmax()]
    return new


def kmeans(X, centres0, max_iter: int = 100):
    """K-means from the starting centres (p. 53): alternate the nearest-centre assignment and the mean update until
    the assignment stops changing. Returns the centres, the assignment and the number of iterations."""
    centres = np.array(centres0, dtype=float)
    assignment = None
    for it in range(max_iter):
        new_assignment = nearest_centre(X, centres)
        if assignment is not None and np.array_equal(new_assignment, assignment):
            break
        assignment = new_assignment
        centres = kmeans_update(X, assignment, centres)
    return centres, assignment, it


def assignment_energy(X, centres, assignment) -> float:
    """E(V, c) = - sum_{p,g} V_pg x_p^T c_g (p. 55)."""
    return -float(np.sum(X * centres[assignment]))


def kmeans_energy(X, centres, assignment) -> float:
    """The energy k-means minimises, sum_{p,g} V_pg |x_p - c_g|^2 (Lecture 8, Homework 1)."""
    return float(np.sum((X - centres[assignment]) ** 2))


def partition_agreement(a, b, k: int) -> float:
    """The fraction of pixels given the same cluster by two assignments, after matching the cluster labels of the two
    runs by the best permutation (the labels of two runs are arbitrary). Exact for k <= 8, greedy beyond."""
    confusion = np.zeros((k, k))
    np.add.at(confusion, (a, b), 1)
    if k <= 8:
        best = max(sum(confusion[g, perm[g]] for g in range(k)) for perm in itertools.permutations(range(k)))
    else:
        best, used = 0.0, set()
        for g in np.argsort(-confusion.max(1)):
            h = max((h for h in range(k) if h not in used), key=lambda h: confusion[g, h])
            best += confusion[g, h]
            used.add(h)
    return best / len(a)


# ----------------------------------------------------------------------------------------------- the two normalisations
def log_affinities(X, centres, sigma: float):
    """The logarithm of the Gaussian affinities e^{-|x_p - c_g|^2 / 2 sigma^2} (p. 58): one row per pixel p, one
    column per centre g."""
    d2 = (X ** 2).sum(1)[:, None] - 2 * X @ centres.T + (centres ** 2).sum(1)[None, :]
    return -np.maximum(d2, 0) / (2 * sigma ** 2)


def soft_assignment_over_clusters(logA):
    """S_pg: the affinities normalised over the clusters (teal, p. 58): for every pixel the N values sum to 1.
    The largest log-affinity of each pixel is subtracted first, so that the exponentials cannot overflow or all
    underflow."""
    S = np.exp(logA - logA.max(axis=1, keepdims=True))
    return S / S.sum(axis=1, keepdims=True)


def assignment_over_pixels(logA):
    """a_gp: the same affinities normalised over the pixels (orange, p. 56): for every centre the HW values sum
    to 1. Returned with the same layout as logA, one column per centre."""
    A = np.exp(logA - logA.max(axis=0, keepdims=True))
    return A / A.sum(axis=0, keepdims=True)


# ----------------------------------------------------------------------------------------------- k-means cross-attention
def kmeans_iterations(X, centres0, n_iter: int = 6):
    """(i) n_iter iterations of k-means: nearest-centre assignment, then every centre is replaced by the mean of its
    pixels. Returns the assignments after each iteration and both energies after every half-step (after the
    assignment, after the update, ...)."""
    centres = np.array(centres0, dtype=float)
    out = {"assignments": [], "E": [], "kmeans_energy": []}
    for _ in range(n_iter):
        assignment = nearest_centre(X, centres)
        out["assignments"].append(assignment)
        out["E"].append(assignment_energy(X, centres, assignment))
        out["kmeans_energy"].append(kmeans_energy(X, centres, assignment))
        centres = kmeans_update(X, assignment, centres)
        out["E"].append(assignment_energy(X, centres, assignment))
        out["kmeans_energy"].append(kmeans_energy(X, centres, assignment))
    out["centres"] = centres
    return out


def kmeans_cross_attention(X, centres0, n_iter: int = 6, length: float | None = None):
    """(ii) k-means cross-attention (p. 61) with W^Q = W^K = W^V = identity, so q_g = c_g and k_p = v_p = x_p.

    Assignment: V_pg = 1 if g = argmax_g' q_g'^T k_p, else 0 (the argmax over the clusters).
    Update: c_g <- c_g + sum_p V_pg v_p (the sum, not the mean), after which every centre is rescaled to the common
    length `length` (default: the root-mean-square length of the rows of X). kMaX-DeepLab divides nothing by the
    count and leaves the scale of c_g to its learned projections (p. 61) and to the normalisation that follows every
    residual sum of a transformer block (p. 11); this toy has neither.
    An empty cluster keeps its centre. The starting centres are rescaled to the same length first.
    Returns the same dictionary as kmeans_iterations.
    """
    length = np.sqrt((X ** 2).sum(1).mean()) if length is None else length

    def rescale(c):
        return length * c / (np.linalg.norm(c, axis=1, keepdims=True) + 1e-12)

    centres = rescale(np.array(centres0, dtype=float))
    out = {"assignments": [], "E": [], "kmeans_energy": []}
    for _ in range(n_iter):
        assignment = (X @ centres.T).argmax(axis=1)                      # V_pg: argmax over the clusters
        out["assignments"].append(assignment)
        out["E"].append(assignment_energy(X, centres, assignment))
        out["kmeans_energy"].append(kmeans_energy(X, centres, assignment))
        V = np.eye(len(centres))[assignment]                             # V_pg as an HW x N matrix of 0s and 1s
        centres = rescale(centres + V.T @ X)                              # c_g <- c_g + sum_p V_pg v_p, then rescaled
        out["E"].append(assignment_energy(X, centres, assignment))
        out["kmeans_energy"].append(kmeans_energy(X, centres, assignment))
    out["centres"] = centres
    return out


# ----------------------------------------------------------------------------------------------- contrastive toy
def embed(theta):
    """Unit vectors z_i = (cos theta_i, sin theta_i) (Lecture 13, p. 22: the z's have length 1)."""
    return np.stack([np.cos(theta), np.sin(theta)], axis=1)


def contrastive_loss(theta, labels, tau: float):
    """The contrastive loss of Lecture 13, p. 22, for views embedded at the angles theta, and its gradient.

    L = - sum_i 1/(2|N_i| - 1) sum_{j in N(y_i), j != i} log[ exp(z_i . z_j / tau) / sum_{k != i} exp(z_i . z_k / tau) ],
    where N(y_i) holds the views with the label of view i and 2|N_i| - 1 is their number without i itself; labels[i]
    is the label y_i of view i (the index of its image in the self-supervised setting).
    Returns the loss, dL/dtheta and the matrix of similarities z_i . z_k.
    """
    z = embed(theta)
    similarity = z @ z.T
    off = ~np.eye(len(theta), dtype=bool)
    scores = np.where(off, similarity / tau, -np.inf)                    # z_i . z_k / tau; the key k = i is excluded
    top = scores.max(axis=1, keepdims=True)
    log_norm = top[:, 0] + np.log(np.exp(scores - top).sum(axis=1))      # log sum_{k != i} exp(z_i . z_k / tau)
    P = np.exp(scores - log_norm[:, None])                               # softmax over the keys k != i
    positives = (labels[:, None] == labels[None, :]) & off
    weights = positives / positives.sum(axis=1, keepdims=True)           # 1 / (2|N_i| - 1) on the positive keys
    loss = -float(np.sum(weights * (similarity / tau - log_norm[:, None])))
    W = weights - P
    D = np.sin(theta[:, None] - theta[None, :])                          # d(z_i . z_k) / d theta_i = -sin(theta_i - theta_k)
    grad = ((W + W.T) * D).sum(axis=1) / tau
    return loss, grad, similarity


def train_contrastive(theta0, labels, tau: float, step: float = 0.02, n_steps: int = 100):
    """Gradient descent on the angles, theta <- theta - step * dL/dtheta. Returns the final angles and the loss
    before every step and after the last one."""
    theta = np.array(theta0, dtype=float)
    losses = []
    for _ in range(n_steps):
        loss, grad, _ = contrastive_loss(theta, labels, tau)
        losses.append(loss)
        theta = theta - step * grad
    losses.append(contrastive_loss(theta, labels, tau)[0])
    return theta, np.array(losses)


def image_gaps(theta, labels):
    """The angular gaps (degrees) between neighbouring images around the circle, each image placed at the mean
    direction of its views; 360 / n for every gap when the n images are spread evenly."""
    z = embed(theta)
    directions = np.array([z[labels == y].mean(axis=0) for y in np.unique(labels)])
    angles = np.sort(np.arctan2(directions[:, 1], directions[:, 0]))
    return np.degrees(np.diff(np.append(angles, angles[0] + 2 * np.pi)))


# ----------------------------------------------------------------------------------------------- plotting
def grid_ticks(ax, grid, size: int):
    """Axis ticks labelled with the (row, col) coordinates of the patch grid."""
    rows, cols = grid
    ax.set_xticks([(c + 0.5) * size for c in range(0, cols, 5)], [str(c) for c in range(0, cols, 5)])
    ax.set_yticks([(r + 0.5) * size for r in range(0, rows, 5)], [str(r) for r in range(0, rows, 5)])
    ax.set_xlabel("grid column")
    ax.set_ylabel("grid row")


def mark_position(ax, o, size: int = 15, color: str = CYAN):
    """A cyan square on the patch at grid position o = (row, col) (p. 6, Figure 2)."""
    ax.add_patch(Rectangle((o[1] * size, o[0] * size), size, size, facecolor=color, edgecolor="k", lw=0.8))


def show_image(ax, image, title: str = "", ticks: bool = True):
    """A colour image with pixel coordinates on the axes."""
    ax.imshow(np.clip(image, 0, 1))
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")
    if not ticks:
        ax.set_xticks([])
        ax.set_yticks([])


def show_map(ax, amap, o=None, title: str = "", size: int = 15, vmax=None, cmap: str = "magma"):
    """An attention or assignment map as an image, bright = large weight, from 0 to vmax (default: the map's own
    maximum), drawn over the pixel area of the image; the query position o is marked in cyan."""
    extent = (0, amap.shape[1] * size, amap.shape[0] * size, 0)
    im = ax.imshow(amap, cmap=cmap, vmin=0, vmax=amap.max() if vmax is None else vmax, extent=extent)
    if o is not None:
        mark_position(ax, o, size)
    ax.set_title(title, fontsize=10)
    if size > 1:
        grid_ticks(ax, amap.shape, size)
    else:
        ax.set_xlabel("x (pixels)")
        ax.set_ylabel("y (pixels)")
    return im


def paint(assignment, colours, shape):
    """An assignment map painted with one colour per cluster."""
    return np.clip(colours, 0, 1)[assignment].reshape(shape[0], shape[1], 3)


def cluster_colours(pixels, assignment, k: int):
    """The mean colour of the pixels of every cluster (grey for an empty cluster), for painting assignment maps."""
    return np.array([pixels[assignment == g].mean(axis=0) if np.any(assignment == g) else np.full(3, 0.5) for g in range(k)])


def plot_embeddings(ax, theta, labels, title: str = ""):
    """The views on the unit circle, coloured by image; the views with the same label are joined by lines."""
    z = embed(theta)
    colours = plt.cm.tab10(labels % 10)
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="0.75"))
    for y in np.unique(labels):
        for i, j in itertools.combinations(np.where(labels == y)[0], 2):
            ax.plot(z[[i, j], 0], z[[i, j], 1], color=colours[i], lw=1.5)
    ax.scatter(z[:, 0], z[:, 1], c=colours, s=60, edgecolor="k", zorder=3)
    for i in range(len(theta)):
        ax.annotate(str(i), 1.13 * z[i], ha="center", va="center", fontsize=8)
    ax.set_aspect("equal")
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_xlabel(r"$z_{1}$")
    ax.set_ylabel(r"$z_{2}$")
    ax.set_title(title, fontsize=10)

## Self-attention on an image

Attention is a weighted average of feature vectors (pp. 5–9): the output $y_o$ at position $o$ sums the inputs $x_p$ at all positions,

$$y_o = \sum_p w_{op}\, x_p, \qquad \text{convolution: } w_{op} = W_{p-o}, \qquad \text{self-attention: } w_{op} = \frac{\exp(x_o^T x_p)}{\sum_{p'} \exp(x_o^T x_{p'})};$$

the self-attention weight depends on the inputs and sums to 1 over $p$. With queries, keys and values $q_o = W^Q x_o$, $k_p = W^K x_p$, $v_p = W^V x_p$ and the scaled dot product ($d$ the token dimension),

$$a_{op} = \frac{\exp(q_o^T k_p / \sqrt{d})}{\sum_{p'} \exp(q_o^T k_{p'} / \sqrt{d})}, \qquad \sum_p a_{op} = 1, \qquad y_o = \sum_p a_{op}\, v_p.$$

Every query meets every key: $O(n^2 d)$ operations for $n$ tokens (pp. 25–27 count the alternatives).

**The tokens.** $15 \times 15$ patches of the image, $x_p$ the mean colour of patch $p$ ($d = 3$); for the $\sqrt{d}$ experiment, the grey patch reduced to $5 \times 5$ pixels ($d = 25$). `patch_tokens` centres the tokens and scales them to length $\sqrt{d}$, so that $x_o^T x_p = d \cos\angle(x_o, x_p)$. With $W^Q = W^K = I$, the maps show the rows $a_{op}$ of a few query positions $o$ (cyan squares); `scale=False` leaves out the division by $\sqrt{d}$.

In [ ]:
image = load_image("2.jpg")                     # a swallowtail on white flowers; "5.jpg" is a farm under a cloudy sky
# The pixel values come from Pillow's JPEG decoder. The Pillow installed by environment.yml (pip) uses libjpeg-turbo
# on every operating system and gives exactly the numbers of our runs; another decoder (macOS ImageIO, or the IJG
# libjpeg of Anaconda's own Pillow) changes most pixels by one level and a few, at colour edges, by ten or more: the
# d = 25 weights then move in the third decimal and the k-means counts by up to a few hundred pixels, while the maps
# and the True/False checks do not change. Small differences from the numbers of other students are expected.
tokens, grid = patch_tokens(image, size=15, kind="colour")
tokens_grey, _ = patch_tokens(image, size=15, kind="grey")
n, d = tokens.shape
print(f"patch grid {grid[0]} x {grid[1]}: n = {n} tokens; d = {d} for the colour tokens, d = {tokens_grey.shape[1]} for the grey-patch tokens")

positions = {"flower (background)": (3, 25), "leaf (texture)": (1, 5), "wing (object)": (12, 8)}    # (row, col) on the grid

fig, axes = plt.subplots(1, 4, figsize=(19, 4), layout="constrained")
show_image(axes[0], image[:grid[0] * 15, :grid[1] * 15], "image and the three query positions o")
grid_ticks(axes[0], grid, 15)
for name, o in positions.items():
    mark_position(axes[0], o)
    axes[0].annotate(name.split(" ")[0], (o[1] * 15 + 18, o[0] * 15 + 12), color="w", fontsize=9, fontweight="bold",
                     path_effects=[pe.withStroke(linewidth=2.5, foreground="k")])
for ax, (name, o) in zip(axes[1:], positions.items()):
    amap = attention_map(tokens, o, grid)       # a_op, scaled by sqrt(d), d = 3
    show_map(ax, amap, o, rf"$a_{{op}}$ for the {name}, o = {o}; max {amap.max():.4f}")
fig.suptitle(r"self-attention weights of three queries; colour tokens (d = 3), scaled by $\sqrt{d}$")
plt.show()

A = attention_weights(tokens, tokens)           # the full n x n matrix a_op, W^Q = W^K = I
print(f"every row of a_op sums to 1: largest |sum_p a_op - 1| = {np.abs(A.sum(1) - 1).max():.1e}")
print(f"the uniform weight over the n = {n} keys would be 1/n = {1 / n:.4f}")
for name, o in positions.items():
    a = attention_map(tokens, o, grid)
    print(f"{name:20s}: largest weight {a.max():.4f}; {keys_for_90_percent(a):3d} of the {n} keys hold 90 % of the weight")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6.6), layout="constrained")
for j, (name, o) in enumerate(positions.items()):
    for i, scale in enumerate([False, True]):
        amap = attention_map(tokens_grey, o, grid, scale=scale)
        label = r"scaled by $\sqrt{d}$" if scale else "unscaled"
        show_map(axes[i, j], amap, o, f"{name}, d = 25, {label}; max {amap.max():.3f}")
        print(f"{name:20s}, d = 25, {'scaled' if scale else 'unscaled':8s}: largest weight {amap.max():.3f}; "
              f"{keys_for_90_percent(amap):3d} of the {n} keys hold 90 % of the weight")
fig.suptitle("grey-patch tokens (d = 25): the same three queries without (top) and with (bottom) the division by $\\sqrt{d}$")
plt.show()

In [ ]:
o_wing = positions["wing (object)"]
row = o_wing[0] * grid[1] + o_wing[1]
fig, ax = plt.subplots(figsize=(7, 6), layout="constrained")
im = ax.imshow(A, cmap="Purples", vmin=0, vmax=np.percentile(A, 99))
ax.axhline(row, color=ORANGE, lw=1.2)
ax.set_xlabel("key p (the grid in raster order: row 0, then row 1, ...)")
ax.set_ylabel("query o")
ax.set_title(rf"the $n \times n$ weights $a_{{op}}$ (colour tokens, scaled); orange: row o = {row}, the wing query")
fig.colorbar(im, ax=ax, fraction=0.046, label="weight (colour scale clipped at the 99th percentile)")
plt.show()
print(f"n = {n} tokens on the {grid[0]} x {grid[1]} grid: the matrix has one row per query and one column per key; each row is one attention map.")

### Question 11.1 (3 points)

- Show the attention maps (colour tokens, $d = 3$) for three positions $o$ of your choice, on the background, a textured region and the object (`positions`, or your own), and describe which positions receive large weights for each query and why, from the formula for $a_{op}$. **(1 point)**
- For the grey-patch tokens ($d = 25$), compare the maps without and with the division by $\sqrt{d}$ for your three queries, reporting the largest weight and the number of keys that hold 90 % of the weight. What does the scaling prevent? **(1 point)**
- For a $64 \times 64$ feature map, compare the operation counts $O(H^2 W^2)$ of global self-attention, $O(HWm^2)$ of local self-attention in an $m \times m$ square and $O(HWm)$ of axial attention with $m = 7$, and say what axial attention gains. **(1 point)**

In [ ]:
# Your code for Question 11.1

**Your answers for Question 11.1:**

## The two normalisations: over the clusters and over the pixels

**k-means on the pixel colours (pp. 53–59)** alternates an assignment and an update of the $k$ centres $c_g$,

$$\text{assignment: } V_{pg} = \begin{cases} 1 & \text{if } g = \arg\max_{g'} x_p^T c_{g'} \\ 0 & \text{otherwise,} \end{cases} \qquad\qquad \text{update: } c_g = \frac{\sum_p V_{pg}\, x_p}{\sum_p V_{pg}} ;$$

the most similar centre is the nearest when the centres have equal length, since $|x_p - c_g|^2 = |x_p|^2 - 2\, x_p^T c_g + |c_g|^2$. The assignment has the energy

$$E(V, c) = -\sum_{p, g} V_{pg}\, x_p^T c_g, \qquad P(V) \propto e^{-E(V, c)},$$

normalised along the axis fixed by the constraint on $V$:

$$\text{over the clusters } \Big(\textstyle\sum_g V_{pg} = 1\Big): \; \langle V_{pg} \rangle = \frac{\exp(x_p^T c_g)}{\sum_{g'} \exp(x_p^T c_{g'})}, \qquad\qquad \text{over the pixels } \Big(\textstyle\sum_p V_{pg} = 1\Big): \; \langle V_{pg} \rangle = \frac{\exp(x_p^T c_g)}{\sum_{p'} \exp(x_{p'}^T c_g)} .$$

A mask transformer uses both (p. 45): its mask prediction $\hat m_g(p)$ is normalised over the $N$ queries (teal), its query update $a_{gp}$ over the $HW$ pixels (orange); over the clusters, Gaussian affinities $e^{-|x_p - c_g|^2 / 2\sigma^2}$ give the posterior $S_{pg}$ of a Gaussian mixture with equal priors and covariance $\sigma^2 I$. Below, `kmeans` ($k = 4$) starts from four random pixels; `soft_assignment_over_clusters` and `assignment_over_pixels` normalise the Gaussian affinities (`log_affinities`, $\sigma$ in colour units) the two ways.

In [ ]:
pixels = image.reshape(-1, 3)
mean_colour = pixels.mean(axis=0)
X = pixels - mean_colour                        # x_p: the pixel colours, centred
H, W = image.shape[:2]
k = 4
rng = np.random.default_rng(0)
centres0 = X[rng.choice(len(X), size=k, replace=False)]      # k random pixels: the start of every run below

centres, assignment, n_iterations = kmeans(X, centres0)
counts = np.bincount(assignment, minlength=k)           # the counts can differ slightly with the JPEG decoder (note above)
print(f"k-means with k = {k} converged after {n_iterations} iterations")
for g in range(k):
    print(f"centre {g}: colour {np.round(centres[g] + mean_colour, 3)}, length |c_g| = {np.linalg.norm(centres[g]):.3f}, {counts[g]:,} pixels")
print(f"E(V, c) = {assignment_energy(X, centres, assignment):,.0f};  k-means energy sum V_pg |x_p - c_g|^2 = {kmeans_energy(X, centres, assignment):,.0f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained", gridspec_kw={"width_ratios": [3, 3, 1.3]})
show_image(axes[0], image, "image")
show_image(axes[1], paint(assignment, centres + mean_colour, (H, W)), f"k-means, k = {k}: each pixel painted with its centre")
for g in range(k):
    axes[2].add_patch(Rectangle((0, k - 1 - g + 0.1), 1, 0.8, color=np.clip(centres[g] + mean_colour, 0, 1)))
    axes[2].text(1.15, k - 1 - g + 0.5, f"centre {g}: {counts[g]:,} pixels", va="center")
axes[2].set_xlim(0, 4)
axes[2].set_ylim(0, k)
axes[2].axis("off")
axes[2].set_title("the centres")
plt.show()

In [ ]:
sigma = 0.15                                    # in colour units; other values for Question 11.2
logA = log_affinities(X, centres, sigma)
S = soft_assignment_over_clusters(logA)         # S_pg, teal: for every pixel the k values sum to 1
A_pix = assignment_over_pixels(logA)            # a_gp, orange: for every centre the H W values sum to 1
print(f"over the clusters: largest |sum_g S_pg - 1| over all pixels  = {np.abs(S.sum(1) - 1).max():.1e}")
print(f"over the pixels:   largest |sum_p a_gp - 1| over all centres = {np.abs(A_pix.sum(0) - 1).max():.1e}")
print(f"the uniform weight over the H W = {H * W:,} pixels would be 1/HW = {1 / (H * W):.2e}")
for g in range(k):
    a = np.sort(A_pix[:, g])[::-1]
    top = int(0.01 * len(a))
    print(f"centre {g}: over the pixels, largest weight {a[0]:.2e} = {a[0] * H * W:5.1f} x uniform, and the top 1 % of the pixels "
          f"hold {a[:top].sum():5.1%} of the weight;  over the clusters, {np.mean(S[:, g] > 0.5):5.1%} of the pixels have S_pg > 0.5")
print("\nthe normalisation over the pixels for other values of sigma (averaged over the centres):")
for sig in (0.3, 0.15, 0.05, 0.02):
    A_sig = assignment_over_pixels(log_affinities(X, centres, sig))
    share = np.mean([np.sort(A_sig[:, g])[::-1][:int(0.01 * H * W)].sum() for g in range(k)])
    print(f"sigma = {sig:4.2f}: largest weight = {np.mean(A_sig.max(0)) * H * W:7.1f} x uniform;  the top 1 % of the pixels hold {share:5.1%} of the weight")

fig, axes = plt.subplots(2, k, figsize=(4.4 * k, 6.4), layout="constrained", squeeze=False)
for g in range(k):
    show_map(axes[0, g], A_pix[:, g].reshape(H, W), None, rf"$a_{{gp}}$, centre {g}; max {A_pix[:, g].max():.1e}", size=1)
    show_map(axes[1, g], S[:, g].reshape(H, W), None, rf"$S_{{pg}}$, centre {g}; from 0 to 1", size=1, vmax=1)
    for r in (0, 1):
        axes[r, g].add_patch(Rectangle((8, 8), 40, 40, facecolor=np.clip(centres[g] + mean_colour, 0, 1), edgecolor="w"))
axes[0, 0].set_ylabel("over the pixels:\neach map sums to 1\n\ny (pixels)", color=ORANGE, fontsize=11, fontweight="bold")
axes[1, 0].set_ylabel(f"over the clusters:\nthe {k} maps sum to 1\n\ny (pixels)", color=TEAL, fontsize=11, fontweight="bold")
fig.suptitle(rf"Gaussian affinities $e^{{-|x_p - c_g|^2 / 2\sigma^2}}$ with $\sigma$ = {sigma}, normalised two ways "
             "(bright = large; each map over the pixels is scaled to its own maximum, the maps over the clusters run from 0 to 1)")
plt.show()

### Question 11.2 (3 points)

- Show the $k = 4$ segmentation and the two sets of maps for a $\sigma$ of your choice, and report the two sums you checked (over $g$ for every pixel, over $p$ for every centre). **(1 point)**
- What does the normalisation over the clusters compute, and what the normalisation over the pixels? Why does a softmax over thousands of pixels concentrate its weight on a few of them (use the printed largest weight against $1/HW$, the share of the top 1 % of the pixels, and how they change with $\sigma$)? **(1 point)**
- List the three differences between cross-attention and k-means clustering and say which one the kMaX authors consider the most significant. **(1 point)**

In [ ]:
# Your code for Question 11.2

**Your answers for Question 11.2:**

## k-means cross-attention

A mask transformer's decoder updates its object queries, the centres $c_g$, by cross-attention (softmax over the pixels, pp. 43, 61); kMaX-DeepLab replaces the softmax by an argmax over the clusters, as in k-means:

$$\text{cross-attention: } c_g \leftarrow c_g + \sum_p \frac{\exp(q_g^T k_p)}{\sum_{p'} \exp(q_g^T k_{p'})}\, v_p, \qquad \text{k-means cross-attention: } c_g \leftarrow c_g + \sum_p V_{pg}\, v_p, \quad V_{pg} = 1 \text{ if } g = \arg\max_{g'} q_{g'}^T k_p, \text{ else } 0 .$$

The values of the assigned pixels are summed, not averaged, and added to $c_g$, not put in its place. Both updates run from the same four starting centres with $W^Q = W^K = W^V = I$: (i) **k-means** (`kmeans_iterations`); (ii) **k-means cross-attention** (`kmeans_cross_attention`), whose centres are rescaled to a common length $s$ after every update (the sums would otherwise make them enormous; kMaX leaves the scale to its learned projections). The figure shows the argmax assignment after each of 6 iterations of both updates, $E(V, c)$ after every half-step and the k-means energy $\sum_{p,g} V_{pg}\, |x_p - c_g|^2$.

In [ ]:
n_iter = 6
km = kmeans_iterations(X, centres0, n_iter)                   # (i)  k-means
ca = kmeans_cross_attention(X, centres0, n_iter)              # (ii) k-means cross-attention, centres rescaled to length s
s = np.sqrt((X ** 2).sum(1).mean())
print(f"common length of the centres in (ii): s = {s:.3f};  lengths of the k-means centres after 6 iterations: {np.round(np.linalg.norm(km['centres'], axis=1), 3)}")

fig, axes = plt.subplots(2, n_iter + 1, figsize=(3.1 * (n_iter + 1), 4.8), layout="constrained")
show_image(axes[0, 0], image, "image", ticks=False)
start = nearest_centre(X, centres0)
show_image(axes[1, 0], paint(start, cluster_colours(pixels, start, k), (H, W)), "start: nearest of the 4 random centres", ticks=False)
for t in range(n_iter):
    for r, (run, name) in enumerate([(km, "(i) k-means"), (ca, "(ii) k-means cross-attention")]):
        a = run["assignments"][t]
        show_image(axes[r, t + 1], paint(a, cluster_colours(pixels, a, k), (H, W)), f"{name}\niteration {t + 1}", ticks=False)
        axes[r, t + 1].set_xlabel("")
        axes[r, t + 1].set_ylabel("")
fig.suptitle("the argmax assignment after each iteration (Figure 30 of the notes, p. 66), regions painted with their mean colour")
plt.show()

half_steps = np.arange(1, 2 * n_iter + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
for ax, key, label in [(axes[0], "E", r"$E(V, c) = -\sum_{p,g} V_{pg}\, x_p^T c_g$ (notes p. 55)"),
                       (axes[1], "kmeans_energy", r"$\sum_{p,g} V_{pg}\, |x_p - c_g|^2$ (Lecture 8)")]:
    ax.plot(half_steps, km[key], marker="o", ms=4, label="(i) k-means")
    ax.plot(half_steps, ca[key], marker="s", ms=4, label="(ii) k-means cross-attention")
    ax.set_xlabel("half-step (odd: after the assignment, even: after the update)")
    ax.set_ylabel("energy")
    ax.set_title(label)
    ax.legend()
plt.show()

for run, name in [(km, "(i)  k-means"), (ca, "(ii) k-means cross-attention")]:
    print(f"{name:30s}: E(V, c) after each iteration {np.round(run['E'][1::2], 0)};  "
          f"E decreases at every half-step: {bool(np.all(np.diff(run['E']) <= 1e-6))};  "
          f"k-means energy decreases at every half-step: {bool(np.all(np.diff(run['kmeans_energy']) <= 1e-6))}")
print(f"agreement of the two partitions after {n_iter} iterations: {partition_agreement(km['assignments'][-1], ca['assignments'][-1], k):.1%} of the pixels;  "
      f"(ii) against converged k-means: {partition_agreement(assignment, ca['assignments'][-1], k):.1%};  "
      f"(i) against converged k-means: {partition_agreement(assignment, km['assignments'][-1], k):.1%}")

### Question 11.3 (2 points)

- Show the assignment maps after each of the 6 iterations for both updates and the plot of $E(V, c)$ against the half-step, and report the agreement between the two final partitions. For which of the two updates does $E(V, c)$ decrease at every half-step, and why? **(1 point)**
- The argmax over the clusters has no gradient. How does kMaX-DeepLab train the affinities $q_g^T k_p$ anyway, and why does that work? **(1 point)**

In [ ]:
# Your code for Question 11.3

**Your answers for Question 11.3:**

## A contrastive-learning toy

In contrastive learning (Lecture 13, pp. 20–23) two augmented views of an image are a positive pair and the other images are negatives. For a batch $I$ of $2n$ views of $n$ images, embedded as unit vectors $z_i$,

$$\mathcal{L} = -\sum_{i=1}^{2n} \frac{1}{2|N_i| - 1} \sum_{j \in N(y_i),\, j \neq i} \log \frac{\exp(z_i \cdot z_j / \tau)}{\sum_{k \in I,\, k \neq i} \exp(z_i \cdot z_k / \tau)},$$

with $y_i$ the label of view $i$, $N(y_i)$ the views with that label, $|N_i|$ the number of such images and $\tau$ a temperature; self-supervised, the label is the image.

**The toy.** No network: the two views of each of $n = 6$ images are embedded directly as $z_i = (\cos\theta_i, \sin\theta_i)$, the twelve angles being the parameters; `train_contrastive` runs gradient descent ($\eta = 0.02$, 100 steps) for $\tau = 0.1$ and $\tau = 1$. Each row of the figure: embeddings before and after (views $2i$, $2i + 1$ belong to image $i$), the loss, the similarities; the printed gaps between neighbouring images are all $60°$ when the six are spread evenly.

In [ ]:
n_images = 6
labels = np.repeat(np.arange(n_images), 2)                      # the label of a view is its image: self-supervised
rng = np.random.default_rng(1)
theta0 = (rng.uniform(0, 2 * np.pi, n_images)[:, None] + rng.uniform(-0.2, 0.2, (n_images, 2))).ravel()
print(f"initial gaps between neighbouring images (degrees): {np.round(image_gaps(theta0, labels), 1)}")

_, grad, _ = contrastive_loss(theta0, labels, tau=0.5)           # gradient check by central finite differences
eps = 1e-5
numerical = np.array([(contrastive_loss(theta0 + eps * e, labels, 0.5)[0] - contrastive_loss(theta0 - eps * e, labels, 0.5)[0]) / (2 * eps)
                      for e in np.eye(len(theta0))])
print(f"gradient check: largest |analytic - numerical| = {np.abs(grad - numerical).max():.1e}")

fig, axes = plt.subplots(2, 4, figsize=(18, 8.4), layout="constrained")
for r, tau in enumerate([0.1, 1.0]):
    theta, losses = train_contrastive(theta0, labels, tau, step=0.02, n_steps=100)
    loss, _, similarity = contrastive_loss(theta, labels, tau)       # similarity[i, k] = z_i . z_k after training
    plot_embeddings(axes[r, 0], theta0, labels, rf"$\tau$ = {tau}: before, $\mathcal{{L}}$ = {losses[0]:.3f}")
    plot_embeddings(axes[r, 1], theta, labels, rf"$\tau$ = {tau}: after {len(losses) - 1} steps, $\mathcal{{L}}$ = {loss:.3f}")
    axes[r, 2].plot(losses)
    axes[r, 2].set_xlabel("gradient step")
    axes[r, 2].set_ylabel(r"contrastive loss $\mathcal{L}$")
    axes[r, 2].set_title(rf"$\tau$ = {tau}: loss during gradient descent")
    im = axes[r, 3].imshow(similarity, cmap="RdBu_r", vmin=-1, vmax=1)
    axes[r, 3].set_xticks(np.arange(2 * n_images))
    axes[r, 3].set_yticks(np.arange(2 * n_images))
    axes[r, 3].set_xlabel("view k")
    axes[r, 3].set_ylabel("view i")
    axes[r, 3].set_title(rf"$\tau$ = {tau}: similarities $z_i \cdot z_k$ after training")
    fig.colorbar(im, ax=axes[r, 3], fraction=0.046)
    positive = np.mean([similarity[2 * i, 2 * i + 1] for i in range(n_images)])
    negatives = similarity[labels[:, None] != labels[None, :]]
    gaps = image_gaps(theta, labels)
    print(f"tau = {tau}: loss {losses[0]:.3f} -> {loss:.3f}, decreasing at every step: {bool(np.all(np.diff(losses) <= 1e-9))}; "
          f"mean similarity of the two views of an image {positive:.4f}; largest similarity between views of different images {negatives.max():.3f}")
    print(f"           gaps between neighbouring images (degrees): {np.round(gaps, 1)}, standard deviation {gaps.std():.1f}")
plt.show()

### Question 11.4 (2 points)

- Show the embeddings before and after training for $\tau = 0.1$ and for $\tau = 1$ and report the final loss of each. Which temperature spreads the six images more evenly around the circle **after the cell's 100 gradient steps** (use the printed gaps), and why? **(1 point)**
- For a view $i$ of the batch, name the query, the positive key and the negative keys. What changes in supervised contrastive learning, where several images share a label? **(1 point)**

In [ ]:
# Your code for Question 11.4

**Your answers for Question 11.4:**